# Strings

A string is an immutable sequence of Unicode code points. This notebook covers creation, indexing, slicing, formatting, searching, splitting, joining, and parsing.

## Learning goals

- Explain the difference between code points and encoded bytes.
- Index and slice strings, including negative indices.
- Format values with f-strings.
- Use the main string methods without guessing their return values.
- Parse simple CSV and XML-like records.
- Implement word counting, palindrome checks, and a Caesar cipher.

## 1. Text is Unicode

A Python `str` stores Unicode code points. Encoding converts those code points to bytes; decoding converts bytes back. UTF-8 is the default on modern systems, but explicit encodings are safer.

In [ ]:
text = "café"
print(text)
print(len(text))
print(len(text.encode("utf-8")))
print(ord("A"))
print(chr(65))

The string has four code points but five UTF-8 bytes because `é` requires two bytes. `len(str)` counts code points, not bytes and not grapheme clusters.

## 2. Creating strings

Single quotes, double quotes, and triple quotes all create strings. Triple quotes preserve newlines. Escape sequences insert characters that would otherwise be difficult to type.

In [ ]:
single = "Python"
double = "Python"
multiline = """First line
Second line"""

print(single)
print(double)
print(multiline)

In [ ]:
print("C:\\Users\\ada")
print(r"C:\Users\ada")
print("tab\tseparated")

## 3. Indexing and slicing

Indices start at zero. Negative indices count from the end. A slice uses `start:stop:step`, with `stop` exclusive. Slices are clipped; direct indexing raises `IndexError`.

In [ ]:
word = "Python"
print(word[0])
print(word[-1])
print(word[1:4])
print(word[:3])
print(word[3:])
print(word[::-1])

Strings are immutable. A method such as `upper` returns a new string and leaves the original unchanged.

In [ ]:
word = "python"
upper_word = word.upper()
print(word)
print(upper_word)

try:
    word[0] = "P"
except TypeError as error:
    print(type(error).__name__, error)

## 4. Operators

`+` concatenates, `*` repeats, comparison operators use lexicographic order, and `in` tests for a substring.

In [ ]:
print("AI" + "!")
print("ab" * 3)
print("apple" < "banana")
print("Z" < "a")
print("th" in "python")

Repeated concatenation inside a loop is quadratic. Collect pieces in a list and join once.

In [ ]:
pieces = []
for number in range(5):
    pieces.append(str(number))
print("-".join(pieces))

## 5. Formatting

An f-string evaluates expressions inside the braces and applies an optional format specification. Common specifications are `d` for integers, `f` for fixed-point numbers, `%` for percentages, and `,` for thousands separators.

In [ ]:
name = "Ada"
score = 0.87654

print(f"{name} scored {score:.2%}")
print(f"{name:>10}|")
print(f"{score:,.3f}")
print(f"{42:05d}")

Legacy code may use the `%` operator or `str.format`. Both are worth recognising, but f-strings are usually clearer.

In [ ]:
print("Hello %s, you are %d years old." % ("Ada", 36))  # noqa: UP031
print("Hello {name}, you are {age} years old.".format(name="Ada", age=36))

## 6. Classification methods

These methods return booleans and are useful for validation.

In [ ]:
print("123".isdigit())
print("12.3".isdigit())
print("abc".isalpha())
print("abc123".isalnum())
print("".isspace())
print("   ".isspace())
print("Ada".istitle())

## 7. Case and layout methods

Case methods return a new string. Layout methods pad or trim a string to a requested width.

In [ ]:
text = "  the quick brown fox  "
print(text.strip())
print(text.title())
print(text.upper())
print("42".zfill(5))
print("python".center(12, "-"))
print("python".ljust(10, "."))
print("python".rjust(10, "."))

## 8. Searching and replacing

`find` returns `-1` when the substring is absent. `index` raises `ValueError`. Use `find` when absence is expected and `index` when absence indicates a bug.

In [ ]:
text = "the quick brown fox"
print(text.find("quick"))
print(text.find("slow"))
print(text.index("quick"))
print(text.count("o"))
print(text.startswith("the"))
print(text.endswith("fox"))
print(text.replace("fox", "dog"))

## 9. Splitting and joining

`split()` without arguments splits on arbitrary whitespace and discards empty strings. `split(",")` preserves empty fields. `partition` returns exactly three pieces around the first occurrence of a separator.

In [ ]:
record = "001,ada,1990-01-01"
fields = record.split(",")
print(fields)
print(" | ".join(fields))
print("key=value".partition("="))

## 10. Parsing a CSV line

When the fields contain no commas inside quotes, `split` and tuple unpacking are enough. For real CSV files, use the `csv` module.

In [ ]:
line = "001,ada,1990-01-01"
identifier, name, date_of_birth = line.split(",")
print(f"{name} has id {identifier} and was born on {date_of_birth}")

try:
    identifier, name = line.split(",")
except ValueError as error:
    print(type(error).__name__, error)

## 11. Parsing an XML-like record

String slicing can extract a value from a very simple, well-formed record. The technique is fragile and should not be used for real XML or HTML.

In [ ]:
data = "<student><name>Ada</name><age>36</age></student>"
name_start = data.find("<name>") + len("<name>")
name_end = data.find("</name>")
name = data[name_start:name_end]
print(name)

This code fails on nested tags, attributes, comments, and malformed input. Use `xml.etree.ElementTree` for XML, `json` for JSON, and a dedicated parser for HTML.

## 12. Word counting

A small pipeline normalises case, splits on whitespace, and counts words with a dictionary. Punctuation still needs a decision: strip it, keep it, or tokenise more carefully.

In [ ]:
def word_counts(text):
    counts = {}
    for word in text.lower().split():
        word = word.strip(".,!?;:")
        if word:
            counts[word] = counts.get(word, 0) + 1
    return counts


sentence = "The quick brown fox jumps over the lazy dog. The fox is quick!"
counts = word_counts(sentence)
for word, count in sorted(counts.items(), key=lambda item: (-item[1], item[0])):
    print(f"{word:>8}: {count}")

## 13. Palindrome check

A palindrome reads the same forwards and backwards after normalisation. Compare the cleaned string with its reverse, or use two pointers.

In [ ]:
def is_palindrome(text):
    cleaned = "".join(character.lower() for character in text if character.isalnum())
    return cleaned == cleaned[::-1]


print(is_palindrome("A man, a plan, a canal: Panama"))
print(is_palindrome("Python"))

## 14. Caesar cipher

A Caesar cipher shifts each letter by a fixed amount and leaves other characters unchanged. The modulo operation wraps around the alphabet.

In [ ]:
def caesar_cipher(text, shift):
    result = []
    for character in text:
        if character.isalpha():
            base = ord("A") if character.isupper() else ord("a")
            shifted = (ord(character) - base + shift) % 26
            result.append(chr(base + shifted))
        else:
            result.append(character)
    return "".join(result)


message = "Attack at dawn!"
encoded = caesar_cipher(message, 3)
decoded = caesar_cipher(encoded, -3)
print(encoded)
print(decoded)
assert decoded == message

## Exercises

1. Count the vowels in a string.
2. Reverse a string without slicing.
3. Normalise a name by stripping whitespace and title-casing it.
4. Parse a three-field CSV record into a dictionary and validate the field count.
5. Count word frequencies in `../data/text/hello_world.txt`.
6. Implement a function that returns the longest word in a sentence.
7. Decode a Caesar cipher without knowing the shift by trying all 26 possibilities.
8. Explain why `len(text)` can differ from the number of visible characters.